# HEART PI composite figures — test version
Read-only cached evidence. Run with heart_gpu. No training or inference. Figure 1 is vector-native; Figure 2 uses existing library images (raster).

In [ ]:
from pathlib import Path
import json, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from IPython.display import display
from PIL import Image
PROJECT=Path('/home/ypan/heart_liver/model_train/heart-molecular-pretraining')
GENERAL=Path('/project/yli4/ypan/heart_evaluation/general_audit_10k_v2')
VALIDATION=Path('/home/ypan/heart_liver/model_train/validation')
RUN=GENERAL/'checkpoint_evaluation/full_10k_batch8'
TRAIN=GENERAL/'checkpoint_evaluation/train_10k_batch8'
BENCH=GENERAL/'chemistry_slide_evidence/independent_property_benchmarks/benchmark_all_runs.csv'
UGI=VALIDATION/'evidence/ugi/ugi_priority1_interactive_evidence'
LNP=VALIDATION/'evidence/lnpdb/ugi_lnpdb_priority2_evidence'
OUT=PROJECT/'summary_figures/heart_pi_composites_v1'
OUT.mkdir(parents=True,exist_ok=True)
plt.rcParams.update({'svg.fonttype':'none','font.family':'DejaVu Sans','font.size':10,'axes.spines.top':False,'axes.spines.right':False})
BLUE='#287da6'; GOLD='#D99B46'
def require(p):
    if not p.is_file(): raise FileNotFoundError(p)
    return p
def read(p): return json.loads(require(p).read_text())
def norm(x):return re.sub('[^a-z0-9]','',str(x).lower())
def col(df,*names):
    lookup={norm(c):c for c in df.columns}
    for name in names:
        if norm(name) in lookup:return lookup[norm(name)]
    raise ValueError(f'Missing {names}; available {list(df.columns)}')
def num(df,*names):return pd.to_numeric(df[col(df,*names)],errors='coerce')
def pct(x):
    x=np.asarray(x,dtype=float)
    return 100*x if np.nanmax(x)<=1.000001 else x
def panel(ax,letter,title):
    ax.set_title(title,fontsize=11);ax.text(-.10,1.05,letter,transform=ax.transAxes,fontsize=15,fontweight='bold')
def save(fig,stem):
    for ext in ['svg','png']:
        fig.savefig(OUT/f'{stem}.{ext}',bbox_inches='tight',facecolor='white',dpi=320)
    plt.show();plt.close(fig)
    print('Saved:',OUT/f'{stem}.svg')
for p in [RUN/'evaluation_summary.json',TRAIN/'evaluation_summary.json',RUN/'atom_token_class_accuracy.csv',RUN/'molecule_diagnostics.csv',RUN/'objective_batches.csv',RUN/'cls_view0.npy',RUN/'projected_view0.npy',BENCH,UGI/'03_full_molecular_audit.json',UGI/'02_3d_index_census.json',LNP/'lnpdb_unique_with_ugi_overlap.csv',LNP/'summary.json']:
    print('FOUND' if p.is_file() else 'MISSING',p)
print('Output:',OUT)

## Figure 1 — General pretraining
Six main panels, nine plots. PCA alignment must be checked against the original evaluation manifest. No arbitrary manifest fallback.

In [ ]:
meta_candidates=[GENERAL/'validation_10k_seed17/sample_metadata.csv',GENERAL/'chemistry_slide_evidence/sample_metadata.csv',RUN/'sample_metadata.csv']
META=next((p for p in meta_candidates if p.is_file()),None) # change explicitly if necessary
print('PCA manifest:',META)
if META is None:raise FileNotFoundError(meta_candidates)
meta=pd.read_csv(META,keep_default_na=False)
if 'sample_order' not in meta:raise ValueError('Missing sample_order; verify alignment before plotting')
order=pd.to_numeric(meta.sample_order,errors='raise')
if order.duplicated().any():raise ValueError('Duplicate sample_order')
meta=meta.assign(sample_order=order).sort_values('sample_order',kind='stable').reset_index(drop=True)
arrs=[np.load(require(RUN/f),mmap_mode='r') for f in ['cls_view0.npy','projected_view0.npy']]
if not all(a.ndim==2 and len(a)==len(meta) for a in arrs):raise ValueError('Embedding/metadata row mismatch')
if not {'heavy_atoms','cLogP'}<=set(meta.columns):
    from rdkit import Chem
    from rdkit.Chem import Descriptors
    ms=[Chem.MolFromSmiles(s) for s in meta.smiles.astype(str)]
    meta['heavy_atoms']=[m.GetNumHeavyAtoms() if m is not None else np.nan for m in ms]
    meta['cLogP']=[Descriptors.MolLogP(m) if m is not None else np.nan for m in ms]
v=read(RUN/'evaluation_summary.json');t=read(TRAIN/'evaluation_summary.json')
atom=pd.read_csv(require(RUN/'atom_token_class_accuracy.csv'))
mol=pd.read_csv(require(RUN/'molecule_diagnostics.csv'));batch=pd.read_csv(require(RUN/'objective_batches.csv'))
fig=plt.figure(figsize=(16,13),layout='constrained');gs=fig.add_gridspec(3,2,height_ratios=[1,1.65,1])
a=fig.add_subplot(gs[0,0]);b=fig.add_subplot(gs[0,1]);x=np.arange(2)
for i,(key,label,color) in enumerate([('masked_atom_accuracy','Pretrained model',BLUE),('masked_atom_majority_class_baseline_accuracy','Majority-class baseline',GOLD)]):
    vals=pct([t[key],v[key]]);bars=a.bar(x+(i-.5)*.34,vals,.34,label=label,color=color);a.bar_label(bars,fmt='%.1f%%',padding=3)
a.set(xticks=x,xticklabels=['Training 10K','Validation 10K'],ylim=(0,115),ylabel='Accuracy (%)');a.legend(frameon=False,loc='lower right');panel(a,'a','Masked atom reconstruction')
vals=[float(v['coord_noisy_rmse']),float(v['coord_pred_rmse'])]
if vals[0]<=0:raise ValueError('Invalid noisy RMSE')
bars=b.bar(['Noisy input','Denoised output'],vals,color=[GOLD,BLUE],width=.55);b.bar_label(bars,fmt='%.4f',padding=3);b.set(ylabel='Aggregate coordinate RMSE (Å)',ylim=(0,max(vals)*1.25));b.text(.98,.90,f'{100*(1-vals[1]/vals[0]):.1f}% error reduction',transform=b.transAxes,ha='right');panel(b,'b','3D coordinate denoising — validation 10K')
sub=gs[1,0].subgridspec(2,2)
for i,(label,arr) in enumerate(zip(['CLS','Contrastive projection'],arrs)):
    pca=PCA(n_components=2,svd_solver='randomized',random_state=17);xy=pca.fit_transform(np.asarray(arr,dtype=np.float32))
    for j,prop in enumerate(['heavy_atoms','cLogP']):
        ax=fig.add_subplot(sub[i,j]);z=pd.to_numeric(meta[prop],errors='coerce').to_numpy();ok=np.isfinite(z)
        if not ok.any():raise ValueError(f'No valid {prop}')
        sc=ax.scatter(xy[ok,0],xy[ok,1],c=z[ok],s=2,alpha=.6,cmap='viridis',linewidths=0,rasterized=False)
        ax.set(title=f'{label} — {prop}',xlabel=f'PC1 ({pca.explained_variance_ratio_[0]:.1%})',ylabel=f'PC2 ({pca.explained_variance_ratio_[1]:.1%})');ax.title.set_fontsize(9);fig.colorbar(sc,ax=ax,shrink=.75)
        if i==j==0:ax.text(-.18,1.16,'c',transform=ax.transAxes,fontsize=15,fontweight='bold')
d=fig.add_subplot(gs[1,1]);sym=col(atom,'token_symbol','element');n=num(atom,'masked_targets','n','count');acc=num(atom,'accuracy','masked_atom_accuracy');ok=(n>0)&acc.notna();bars=d.bar(atom.loc[ok,sym].astype(str),pct(acc[ok]),color=BLUE);d.bar_label(bars,labels=[f'{int(k):,}' for k in n[ok]],padding=3,fontsize=8);d.set(ylabel='Recovery accuracy (%)',ylim=(0,115),xlabel='Element (labels above bars = masked targets)');panel(d,'d','Per-element masked atom recovery')
e=fig.add_subplot(gs[2,0]);x=num(mol,'coord_noisy_rmse');y=num(mol,'coord_pred_rmse');ok=x.notna()&y.notna()&(x>=0)&(y>=0)
if not ok.any():raise ValueError('No valid coordinate pairs')
e.scatter(x[ok],y[ok],s=5,alpha=.25,color=BLUE);mx=max(x[ok].max(),y[ok].max());e.plot([0,mx],[0,mx],'--',color='gray',lw=1);e.set(xlabel='Noisy input RMSE (Å)',ylabel='Denoised RMSE (Å)');panel(e,'e','Coordinate recovery per view')
f=fig.add_subplot(gs[2,1]);z=num(batch,'contrastive_loss','nt_xent_loss').dropna();f.hist(z,bins=30,color=BLUE);f.set(xlabel='Recorded NT-Xent loss',ylabel='Number of batches');panel(f,'f','Contrastive loss across batches')
fig.suptitle('HEART general pretraining validation',fontsize=18,fontweight='bold');save(fig,'HEART_Figure1_general_pretraining')
print('PCA row count/order checked; confirm identity alignment against original evaluation.')

## Figure 2 — Downstream and virtual library
First list the actual library images. Set overrides if filenames are ambiguous. The top-row crop must be visually checked. Library panels are raster images inside SVG, not editable vector plots.

In [ ]:
for root in [UGI/'figures',LNP/'figures']:
    print(root)
    for p in sorted(root.glob('*')):print(' ',p.name)
UGI_TOP_IMAGE=None
CONFORMER_IMAGE=None
OVERLAP_IMAGE=None
# Replace None with exact full paths if needed.
def choose(root,override,terms):
    if override is not None:return require(Path(override))
    files=[p for p in root.glob('*.png') if all(t in p.name.lower() for t in terms)]
    if len(files)!=1:raise ValueError(f'Set exact image path: {root}; search={terms}; matches={files}')
    return files[0]
ugi=choose(UGI/'figures',UGI_TOP_IMAGE,['readiness']);conf=choose(UGI/'figures',CONFORMER_IMAGE,['real','conformer']);overlap=choose(LNP/'figures',OVERLAP_IMAGE,['overlap'])
print('Selected:',ugi,conf,overlap)

In [ ]:
def canonical_benchmark(value):
    return {'esol':'ESOL','delaney':'ESOL','freesolv':'FreeSolv','sampl':'FreeSolv','lipophilicity':'Lipophilicity','lipo':'Lipophilicity'}.get(norm(value))
def canonical_method(value):
    s=norm(value)
    if 'morgan' in s or 'ecfp' in s:return 'Morgan ECFP4'
    if 'rdkit' in s:return 'RDKit descriptors'
    if 'heavy' in s:return 'Heavy atoms only'
    if 'mean' in s or 'dummy' in s:return 'Mean label'
    if any(t in s for t in ['frozen','heart','pretrain','cls']):return 'Frozen pretrained CLS'
    return str(value).strip()
raw=pd.read_csv(require(BENCH));bc=col(raw,'benchmark','dataset','task');mc=col(raw,'model','method');fc=col(raw,'train_fraction','fraction');rc=col(raw,'rmse','test_rmse');sc=col(raw,'seed')
q=raw.assign(bench=raw[bc].map(canonical_benchmark),method=raw[mc].map(canonical_method),rmse=pd.to_numeric(raw[rc],errors='coerce'))
q=q[np.isclose(pd.to_numeric(q[fc],errors='coerce'),1.0)&q.bench.notna()&q.rmse.notna()].copy()
if q.duplicated(['bench','method',sc]).any():raise ValueError('Repeated seed rows; review before plotting')
agg=q.groupby(['bench','method']).rmse.agg(['mean','std','count']);display(agg)
fig=plt.figure(figsize=(17,12),layout='constrained');gs=fig.add_gridspec(3,3,height_ratios=[1,1,1.2]);methods=['Frozen pretrained CLS','RDKit descriptors','Morgan ECFP4','Heavy atoms only','Mean label']
for i,name in enumerate(['FreeSolv','ESOL','Lipophilicity']):
    ax=fig.add_subplot(gs[0,i]);s=agg.loc[name].reindex(methods)
    if s['mean'].isna().any():raise ValueError(f'Missing benchmark methods: {name}')
    y=np.arange(len(s));ax.barh(y,s['mean'],xerr=s['std'].fillna(0),capsize=3,color=[BLUE]+['#a6b2bf']*4);ax.set_yticks(y,methods,fontsize=9);ax.invert_yaxis();ax.set_xlabel('Scaffold-held-out RMSE');ax.set_xlim(0,max(s['mean']+s['std'].fillna(0))*1.45)
    for yy,(_,r) in enumerate(s.iterrows()):
        sd=r['std'];err=sd if np.isfinite(sd) else 0
        ax.text(r['mean']+err+.02,yy,f"{r['mean']:.3f} ± {sd:.3f}" if np.isfinite(sd) else f"{r['mean']:.3f} (n=1)",va='center',fontsize=8)
    panel(ax,'abc'[i],name)
im=Image.open(ugi).convert('RGB');w,h=im.size;UGI_TOP_CROP=(0,0,w,int(h*.53)) # inspect visually
ax=fig.add_subplot(gs[1,:]);ax.imshow(im.crop(UGI_TOP_CROP));ax.axis('off');ax.set_title('d  Ugi-4CR library: conformer count, molecular size, canonical identity')
for spec,path,title in [(gs[2,0],conf,'e  Representative stored 3D conformer'),(gs[2,1:],overlap,'f  LNPDB vs. full Ugi-4CR exact molecular overlap')]:
    ax=fig.add_subplot(spec);ax.imshow(Image.open(path).convert('RGB'));ax.axis('off');ax.set_title(title)
fig.suptitle('HEART downstream transfer and lipid-library characterization',fontsize=18,fontweight='bold');save(fig,'HEART_Figure2_downstream_and_library')
(OUT/'figure2_sources.json').write_text(json.dumps({'benchmark':str(BENCH),'ugi_image':str(ugi),'ugi_crop':UGI_TOP_CROP,'conformer_image':str(conf),'overlap_image':str(overlap),'library_panels':'embedded raster images'},indent=2))